# Notebook 4: New Sample Prediction & Recommendations

In this notebook I apply the trained KMeans model to the test dataset (the user's listening history).
After assigning clusters to those songs, I'll recommend new songs from the same clusters.

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# Load the test dataset from the data folder
df_test = pd.read_csv('../data/recommend - recommend.csv')
print("Test data shape:", df_test.shape)
df_test.head()

## 1. Preprocess the Test Data

I apply the same preprocessing steps as the training data:
- Drop the same non-numeric and identifier columns
- Drop the same highly correlated columns that were dropped during training
- Scale the data

In [16]:
# Save test identifiers
id_cols = ['artist_name', 'track_name', 'release_date', 'genre', 'topic']
# topic may not exist in test — handle gracefully
id_cols_present = [c for c in id_cols if c in df_test.columns]
test_ids = df_test[id_cols_present].copy()

# Drop same columns as training
cols_to_drop = ['Unnamed: 0', 'lyrics', 'artist_name', 'track_name',
                'release_date', 'genre', 'topic']
cols_to_drop_present = [c for c in cols_to_drop if c in df_test.columns]
df_test_clean = df_test.drop(columns=cols_to_drop_present)

print("Test data shape after dropping:", df_test_clean.shape)
print("Columns:", df_test_clean.columns.tolist())

Test data shape after dropping: (10, 18)
Columns: ['len', 'dating', 'violence', 'world/life', 'night/time', 'shake the audience', 'family/gospel', 'romantic', 'communication', 'obscene', 'music', 'movement/places', 'light/visual perceptions', 'family/spiritual', 'like/girls', 'sadness', 'feelings', 'age']


In [ ]:
# Load the training cleaned data to get the exact feature set used
df_train_cleaned = pd.read_csv('../data/train_cleaned.csv')
train_features = df_train_cleaned.columns.tolist()
print("Training features used:", train_features)

# Keep only the columns that were used during training
# Some columns (like 'like/girls') may appear in test but not train — drop those
df_test_clean = df_test_clean[train_features]
print("Test data shape after aligning features:", df_test_clean.shape)

In [ ]:
# Load raw training data to fit the scaler correctly
# We need unscaled data so the scaler learns the right mean and std from training
df_train_raw = pd.read_csv('../train.csv')
cols_to_drop_raw = ['Unnamed: 0', 'lyrics', 'artist_name', 'track_name',
                    'release_date', 'genre', 'topic']
df_train_raw = df_train_raw.drop(columns=[c for c in cols_to_drop_raw if c in df_train_raw.columns])
df_train_raw = df_train_raw[train_features]

# Fit on training data, then transform test data — ensures consistent scaling
scaler = StandardScaler()
scaler.fit(df_train_raw)
df_test_scaled = scaler.transform(df_test_clean)
df_test_scaled = pd.DataFrame(df_test_scaled, columns=train_features)

print("Scaled test data shape:", df_test_scaled.shape)
df_test_scaled.head()

## 2. Apply the Trained KMeans Model

In [ ]:
# Load the trained KMeans model saved from notebook 3
with open('../data/kmeans_model.pkl', 'rb') as f:
    kmeans_model = pickle.load(f)

# Predict which cluster each test song belongs to
test_cluster_labels = kmeans_model.predict(df_test_scaled)

# Attach cluster labels to the song identifiers dataframe
test_ids = test_ids.reset_index(drop=True)
test_ids['cluster'] = test_cluster_labels

print("Cluster assignments for the user's songs:")
print(test_ids[['artist_name', 'track_name', 'cluster']].to_string(index=False))

## 3. Generate Recommendations

Now I look at which clusters the user's songs belong to,
and find other songs from the training set in those same clusters.
Those are the recommendations!

In [ ]:
# Load the training set with cluster labels — this is our recommendation pool
df_train_clusters = pd.read_csv('../data/train_with_clusters.csv')

# Find which clusters the user's songs belong to
user_clusters = test_ids['cluster'].unique()
print("User's songs fall into these clusters:", user_clusters)

# Build a set of the user's track names to avoid recommending what they already have
user_tracks = set(test_ids['track_name'].str.lower())

recommendations = []
for cluster in user_clusters:
    cluster_songs = df_train_clusters[df_train_clusters['cluster'] == cluster]
    # Exclude songs already in the user's listening history
    cluster_songs = cluster_songs[
        ~cluster_songs['track_name'].str.lower().isin(user_tracks)
    ]
    # Sample 3 songs per cluster as recommendations
    sample = cluster_songs[['artist_name', 'track_name', 'genre', 'cluster']].sample(
        min(3, len(cluster_songs)), random_state=42
    )
    recommendations.append(sample)

df_recommendations = pd.concat(recommendations, ignore_index=True)
print("\nRecommended songs for this user:")
print(df_recommendations.to_string(index=False))

## 4. Save Results

In [ ]:
# Save test songs with their assigned cluster labels
test_ids.to_csv('../data/test_with_clusters.csv', index=False)
print("Saved test_with_clusters.csv")

# Save the recommended songs for reporting
df_recommendations.to_csv('../data/recommendations.csv', index=False)
print("Saved recommendations.csv")

print("\nFinal recommendation summary:")
print(df_recommendations[['artist_name', 'track_name', 'genre']].to_string(index=False))